# Extended Thinking

Extended thinking gives Claude "scratch paper" — time to reason through a hard problem before producing the final answer. The response gains a **thinking block** (the reasoning) alongside the usual **text block** (the answer).

**Benefits:** better reasoning, higher accuracy on hard problems, transparency into the thought process.
**Trade-offs:** you pay for thinking tokens, higher latency, and more complex response handling.

**When to use it:** *eval-driven*. Run without thinking first; only if accuracy still falls short **after** you've optimized the prompt should you turn it on. It's a lever for when standard prompting isn't enough — not a default.

> ⚠️ **Model-drift note (this is the big one for this section).** How you *enable* thinking depends on the model:
> - **Legacy shape — `{"type": "enabled", "budget_tokens": N}`** — used by **Claude Haiku 4.5** (this notebook), Opus 4.5, Sonnet 4.6, and earlier Claude 4 models. This is exactly what the course shows, so it works verbatim here.
> - **Adaptive shape — `{"type": "adaptive"}`** — used by the newest flagships (**Sonnet 5, Opus 4.8/4.7, Fable 5**). On those, thinking is always-on and depth is set by a separate `effort` param — and passing `budget_tokens` returns a **400**.
>
> **Compatibility:** extended thinking is **not** compatible with assistant-response **prefilling**, and requires **temperature = 1** (we keep the default `1.0`). We use `claude-haiku-4-5-20251001`.

## Setup

`chat` gains `thinking` / `thinking_budget`. `max_tokens` must exceed `thinking_budget` (min budget is 1024). The `thinking_test_str` magic string forces a *redacted* thinking block for testing.

In [1]:
from dotenv import load_dotenv, find_dotenv
load_dotenv(find_dotenv())

from anthropic import Anthropic
from anthropic.types import Message

client = Anthropic()
model = "claude-haiku-4-5-20251001"

# Magic string that forces Claude to return a redacted_thinking block (for testing)
thinking_test_str = "ANTHROPIC_MAGIC_STRING_TRIGGER_REDACTED_THINKING_46C9A13E193C177646C7398A98432ECCCE4C1253D5E2D82641AC0E52CC2876CB"


def add_user_message(messages, message):
    messages.append({
        "role": "user",
        "content": message.content if isinstance(message, Message) else message,
    })


def add_assistant_message(messages, message):
    messages.append({
        "role": "assistant",
        "content": message.content if isinstance(message, Message) else message,
    })


def chat(messages, system=None, temperature=1.0, stop_sequences=[],
         tools=None, thinking=False, thinking_budget=1024):
    params = {
        "model": model,
        "max_tokens": 4000,           # must be > thinking_budget
        "messages": messages,
        "temperature": temperature,   # must be 1 when thinking is enabled
        "stop_sequences": stop_sequences,
    }

    if thinking:
        # Legacy shape — correct for Haiku 4.5. Flagships (Sonnet 5/Opus 4.8) would 400
        # on budget_tokens and instead use {"type": "adaptive"} + the effort param.
        params["thinking"] = {
            "type": "enabled",
            "budget_tokens": thinking_budget,
        }

    if tools:
        params["tools"] = tools
    if system:
        params["system"] = system

    message = client.messages.create(**params)
    return message


def text_from_message(message):
    return "\n".join([block.text for block in message.content if block.type == "text"])


print(f"Ready. Model: {model}")

Ready. Model: claude-haiku-4-5-20251001


## A helper to render every block type

With thinking on, `message.content` can hold `thinking`, `redacted_thinking`, and `text` blocks. This walks them so we can see the reasoning, note any redaction, and read the answer.

In [2]:
def render_thinking_response(message):
    for block in message.content:
        if block.type == "thinking":
            print("=== THINKING ===")
            print(block.thinking)
            print(f"\n[signature present: {bool(getattr(block, 'signature', None))}]")
        elif block.type == "redacted_thinking":
            print("=== REDACTED THINKING ===")
            print("[encrypted reasoning — not human-readable; preserve unmodified for multi-turn context]")
        elif block.type == "text":
            print("\n=== ANSWER ===")
            print(block.text)
        print()

## Thinking in action

A question where a moment's reasoning prevents the intuitive-but-wrong answer. With thinking enabled you'll see Claude work it out, then answer.

In [3]:
messages = []
add_user_message(
    messages,
    "A bat and a ball cost $1.10 in total. The bat costs $1.00 more than the ball. "
    "How much does the ball cost? Think it through carefully.",
)

response = chat(messages, thinking=True, thinking_budget=2000)
render_thinking_response(response)

=== THINKING ===
Let me denote the cost of the ball as $x.

If the bat costs $1.00 more than the ball, then the bat costs $x + 1.00.

The total cost is $1.10, so:
x + (x + 1.00) = 1.10

Simplifying:
2x + 1.00 = 1.10
2x = 1.10 - 1.00
2x = 0.10
x = 0.05

So the ball costs $0.05 (or 5 cents).

Let me verify: 
- Ball costs $0.05
- Bat costs $0.05 + $1.00 = $1.05
- Total: $0.05 + $1.05 = $1.10 ✓

This is correct.

[signature present: True]


=== ANSWER ===
# Solution

Let me set up an equation where the ball's cost = **x**

**Given information:**
- Ball + Bat = $1.10
- Bat = Ball + $1.00

**Setting up the equation:**
- x + (x + 1.00) = 1.10
- 2x + 1.00 = 1.10
- 2x = 0.10
- x = 0.05

**Answer: The ball costs $0.05 (5 cents)**

**Verification:**
- Ball: $0.05
- Bat: $0.05 + $1.00 = $1.05
- Total: $0.05 + $1.05 = $1.10 ✓



## Inspect the raw block types

Confirm the structure: a `thinking` block (summarized reasoning + a `signature`) followed by a `text` block. On Haiku 4.5 thinking `display` defaults to `"summarized"`, so `block.thinking` is a summary of the full reasoning; the **signature** encrypts the *full* thinking.

In [4]:
for block in response.content:
    print(block.type)

thinking
text


## Signatures & multi-turn

The `signature` is a cryptographic token over the full thinking content — it lets the server verify you haven't tampered with the reasoning (tampering could steer the model unsafely). When you continue a conversation, pass thinking blocks back **unchanged, signature included**; our `add_assistant_message` preserves the whole block list, so that's handled.

## Redacted thinking

Sometimes Claude's reasoning is flagged by internal safety systems and returned **encrypted** as a `redacted_thinking` block instead of readable text. You can't read it, but you must **pass it back unmodified** so context isn't lost in multi-turn. The magic trigger string forces one so you can verify your code handles it without crashing.

In [5]:
messages = []
add_user_message(messages, thinking_test_str)

redacted_response = chat(messages, thinking=True)
render_thinking_response(redacted_response)

=== REDACTED THINKING ===
[encrypted reasoning — not human-readable; preserve unmodified for multi-turn context]


=== ANSWER ===
I don't have special "magic strings" or hidden commands that unlock different behaviors or modes. That string doesn't do anything special with me.

I'm Claude, made by Anthropic. I work the same way regardless of what text is input - I aim to be helpful, harmless, and honest in all my responses.

Is there something I can actually help you with today?



In [6]:
# Confirm we actually got a redacted_thinking block
for block in redacted_response.content:
    print(block.type)

redacted_thinking
text


## Takeaway

Extended thinking trades **cost + latency** for **reasoning quality**. Reach for it only when your evals show standard prompting isn't enough. Handle the response defensively: expect `thinking` (or `redacted_thinking`) blocks *before* the answer, preserve them (and their signatures) unmodified across turns, and extract the `text` block for display.

> **Model reminder:** the `{"type": "enabled", "budget_tokens": N}` shape used here is correct for Haiku 4.5; on Sonnet 5 / Opus 4.8 you'd switch to `{"type": "adaptive"}` and control depth with `effort` (see the README adaptations log).

Next: **Image support** — sending images to Claude.